# 🎌 Next Anime — Sistem Rekomendasi Anime
## Notebook: EDA & Preprocessing
Dataset: MAL API + Kaggle MyAnimeList Dataset 2023

**Tujuan:** Membangun dataset bersih sebagai fondasi sistem rekomendasi anime berbasis Content-Based Filtering menggunakan TF-IDF + Cosine Similarity.

## 1. 📦 Import Library & Load Dataset
Mengimport library yang dibutuhkan dan memuat dua dataset:
- `anime_dataset.csv` — data dari MAL API (2.000 anime top rating)
- `anime-dataset-2023.csv` — data dari Kaggle (24.905 anime)

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load kedua dataset
df_mal = pd.read_csv('../data/anime_dataset.csv')
df_kaggle = pd.read_csv('../data/anime-dataset-2023.csv')

print("MAL API Dataset:", df_mal.shape)
print("Kaggle Dataset:", df_kaggle.shape)
print("\nMAL Columns:", df_mal.columns.tolist())
print("\nKaggle Columns:", df_kaggle.columns.tolist())

MAL API Dataset: (2000, 10)
Kaggle Dataset: (24905, 24)

MAL Columns: ['id', 'title', 'mean', 'rank', 'popularity', 'synopsis', 'media_type', 'num_episodes', 'genres', 'studios']

Kaggle Columns: ['anime_id', 'Name', 'English name', 'Other name', 'Score', 'Genres', 'Synopsis', 'Type', 'Episodes', 'Aired', 'Premiered', 'Status', 'Producers', 'Licensors', 'Studios', 'Source', 'Duration', 'Rating', 'Rank', 'Popularity', 'Favorites', 'Scored By', 'Members', 'Image URL']


## 2. 🔗 Merge Dataset
Menggabungkan kedua dataset menjadi satu, menyamakan nama kolom, dan membuang duplikat berdasarkan judul anime.

In [3]:
# Samakan nama kolom Kaggle agar bisa digabung
df_kaggle_clean = df_kaggle.rename(columns={
    'Name': 'title',
    'Genres': 'genres',
    'Synopsis': 'synopsis',
    'Score': 'mean',
    'Rank': 'rank',
    'Popularity': 'popularity',
    'Type': 'media_type',
    'Episodes': 'num_episodes',
    'Studios': 'studios'
})

# Pilih kolom yang relevan dari Kaggle
cols = ['title', 'mean', 'rank', 'popularity', 'synopsis', 'media_type', 'num_episodes', 'genres', 'studios']
df_kaggle_clean = df_kaggle_clean[cols]

# Gabungkan kedua dataset
df_combined = pd.concat([df_mal, df_kaggle_clean], ignore_index=True)

# Buang duplikat berdasarkan judul
df_combined = df_combined.drop_duplicates(subset='title', keep='first')
df_combined = df_combined.reset_index(drop=True)

print("Total anime setelah digabung:", df_combined.shape)
print("Missing values:\n", df_combined.isnull().sum())

Total anime setelah digabung: (25240, 10)
Missing values:
 id              23240
title               0
mean                0
rank                0
popularity          0
synopsis            4
media_type          0
num_episodes        0
genres              0
studios             8
dtype: int64


## 3. 🧹 Data Cleaning
- Menghapus kolom `id` yang tidak dibutuhkan
- Mengisi missing values pada `synopsis` dan `studios`
- Konversi tipe data numerik

In [4]:
# Drop kolom id (tidak dibutuhkan untuk rekomendasi)
df_combined = df_combined.drop(columns=['id'])

# Isi missing values
df_combined['synopsis'] = df_combined['synopsis'].fillna('')
df_combined['studios'] = df_combined['studios'].fillna('Unknown')

# Konversi tipe data
df_combined['mean'] = pd.to_numeric(df_combined['mean'], errors='coerce').fillna(0)
df_combined['rank'] = pd.to_numeric(df_combined['rank'], errors='coerce').fillna(0)
df_combined['popularity'] = pd.to_numeric(df_combined['popularity'], errors='coerce').fillna(0)
df_combined['num_episodes'] = pd.to_numeric(df_combined['num_episodes'], errors='coerce').fillna(0)

# Cek hasil
print("Shape:", df_combined.shape)
print("\nMissing values:\n", df_combined.isnull().sum())
print("\nSample data:")
df_combined.head(3)

Shape: (25240, 9)

Missing values:
 title           0
mean            0
rank            0
popularity      0
synopsis        0
media_type      0
num_episodes    0
genres          0
studios         0
dtype: int64

Sample data:


,title,mean,rank,popularity,synopsis,media_type,num_episodes,genres,studios
0,Sousou no Frieren,9.26,1.0,104,During their decade-long quest to defeat the D...,tv,28.0,"Adventure, Award Winning, Drama, Fantasy, Shounen",Madhouse
1,Steel Ball Run: JoJo no Kimyou na Bouken,9.13,2.0,1352,"In the American Old West, the world's greatest...",ona,0.0,"Action, Adventure, Historical, Mystery, Racing...",David Production
2,Fullmetal Alchemist: Brotherhood,9.11,3.0,3,After a horrific alchemy experiment goes wrong...,tv,64.0,"Action, Adventure, Drama, Fantasy, Military, S...",Bones


In [5]:
# Fix tipe data ke integer
df_combined['rank'] = df_combined['rank'].astype(int)
df_combined['num_episodes'] = df_combined['num_episodes'].astype(int)
df_combined['popularity'] = df_combined['popularity'].astype(int)

# Cek hasil
print(df_combined.dtypes)
print("\nSample:")
df_combined.head(3)

title            object
mean            float64
rank              int64
popularity        int64
synopsis         object
media_type       object
num_episodes      int64
genres           object
studios          object
dtype: object

Sample:


,title,mean,rank,popularity,synopsis,media_type,num_episodes,genres,studios
0,Sousou no Frieren,9.26,1,104,During their decade-long quest to defeat the D...,tv,28,"Adventure, Award Winning, Drama, Fantasy, Shounen",Madhouse
1,Steel Ball Run: JoJo no Kimyou na Bouken,9.13,2,1352,"In the American Old West, the world's greatest...",ona,0,"Action, Adventure, Historical, Mystery, Racing...",David Production
2,Fullmetal Alchemist: Brotherhood,9.11,3,3,After a horrific alchemy experiment goes wrong...,tv,64,"Action, Adventure, Drama, Fantasy, Military, S...",Bones


## 4. 📊 Exploratory Data Analysis (EDA)
Mengeksplorasi distribusi data untuk memahami karakteristik dataset.

In [6]:
# Distribusi media type
print("=== Distribusi Media Type ===")
print(df_combined['media_type'].value_counts())

print("\n=== Statistik Rating (mean) ===")
print(df_combined[df_combined['mean'] > 0]['mean'].describe())

print("\n=== Top 10 Genre Terpopuler ===")
all_genres = df_combined['genres'].str.split(', ').explode()
print(all_genres.value_counts().head(10))

print("\n=== Total Anime per Kategori ===")
print(f"Anime dengan rating   : {len(df_combined[df_combined['mean'] > 0])}")
print(f"Anime tanpa rating    : {len(df_combined[df_combined['mean'] == 0])}")
print(f"Anime ongoing (0 eps) : {len(df_combined[df_combined['num_episodes'] == 0])}")
print(f"Total anime           : {len(df_combined)}")

=== Distribusi Media Type ===
media_type
TV            6665
Movie         4015
OVA           3919
ONA           3440
Music         2683
Special       2449
tv            1068
movie          446
ona            192
ova            166
special         80
UNKNOWN         69
tv_special      48
Name: count, dtype: int64

=== Statistik Rating (mean) ===
count    16124.000000
mean         6.427969
std          0.958929
min          1.850000
25%          5.750000
50%          6.420000
75%          7.130000
max          9.260000
Name: mean, dtype: float64

=== Top 10 Genre Terpopuler ===
genres
Comedy           7184
Fantasy          5342
UNKNOWN          4888
Action           4797
Adventure        3951
Sci-Fi           3124
Drama            2900
Romance          2105
Slice of Life    1721
Supernatural     1505
Name: count, dtype: int64

=== Total Anime per Kategori ===
Anime dengan rating   : 16124
Anime tanpa rating    : 9116
Anime ongoing (0 eps) : 557
Total anime           : 25240


## 5. ✏️ Standardisasi Data
Menyamakan format penulisan pada kolom `media_type`, `genres`, dan `studios` ke lowercase untuk menghindari inkonsistensi data dari dua sumber berbeda.

In [7]:
# Standardisasi media_type ke lowercase
df_combined['media_type'] = df_combined['media_type'].str.lower().str.strip()

# Standardisasi genres ke lowercase
df_combined['genres'] = df_combined['genres'].str.lower().str.strip()

# Standardisasi studios ke lowercase
df_combined['studios'] = df_combined['studios'].str.lower().str.strip()

# Cek hasil media_type setelah standardisasi
print("=== Media Type setelah standardisasi ===")
print(df_combined['media_type'].value_counts())

=== Media Type setelah standardisasi ===
media_type
tv            7733
movie         4461
ova           4085
ona           3632
music         2683
special       2529
unknown         69
tv_special      48
Name: count, dtype: int64


## 6. 💾 Simpan Dataset Bersih
Menyimpan dataset yang sudah bersih ke file `anime_cleaned.csv` untuk digunakan pada tahap pembuatan model.

In [9]:
# Simpan dataset bersih
df_combined.to_csv('../data/anime_cleaned.csv', index=False)

print("Dataset bersih berhasil disimpan!")
print(f"Total anime: {len(df_combined)}")
print(f"\nSample data:")
df_combined.head()

Dataset bersih berhasil disimpan!
Total anime: 25240

Sample data:


,title,mean,rank,popularity,synopsis,media_type,num_episodes,genres,studios
0,Sousou no Frieren,9.26,1,104,During their decade-long quest to defeat the D...,tv,28,"adventure, award winning, drama, fantasy, shounen",madhouse
1,Steel Ball Run: JoJo no Kimyou na Bouken,9.13,2,1352,"In the American Old West, the world's greatest...",ona,0,"action, adventure, historical, mystery, racing...",david production
2,Fullmetal Alchemist: Brotherhood,9.11,3,3,After a horrific alchemy experiment goes wrong...,tv,64,"action, adventure, drama, fantasy, military, s...",bones
3,Chainsaw Man Movie: Reze-hen,9.07,4,540,Despite the immediate challenges following bec...,movie,1,"action, fantasy, gore, shounen, urban fantasy",mappa
4,Steins;Gate,9.07,5,14,Eccentric scientist Rintarou Okabe has a never...,tv,24,"drama, psychological, sci-fi, suspense, time t...",white fox


In [2]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pickle

# Load dataset yang sudah bersih
df_combined = pd.read_csv('../data/anime_cleaned.csv')
indices_v2 = pd.Series(df_combined.index, index=df_combined['title'].str.lower())

print(f"Dataset loaded! Total anime: {len(df_combined)}")
df_combined.head()

Dataset loaded! Total anime: 25240


,title,mean,rank,popularity,synopsis,media_type,num_episodes,genres,studios,tags,tags_v2,tags_v3,tags_v4
0,Sousou no Frieren,9.26,1,104,During their decade-long quest to defeat the D...,tv,28,"adventure, award winning, drama, fantasy, shounen",madhouse,adventure award winning drama fantasy shou...,adventure award winning drama fantasy shou...,tv tv tv tv tv adventure award winning drama...,tv tv tv tv tv tv tv tv adventure award winni...
1,Steel Ball Run: JoJo no Kimyou na Bouken,9.13,2,1352,"In the American Old West, the world's greatest...",ona,0,"action, adventure, historical, mystery, racing...",david production,action adventure historical mystery racing...,action adventure historical mystery racing...,ona ona ona ona ona action adventure histori...,ona ona ona ona ona ona ona ona action advent...
2,Fullmetal Alchemist: Brotherhood,9.11,3,3,After a horrific alchemy experiment goes wrong...,tv,64,"action, adventure, drama, fantasy, military, s...",bones,action adventure drama fantasy military s...,action adventure drama fantasy military s...,tv tv tv tv tv action adventure drama fanta...,tv tv tv tv tv tv tv tv action adventure dra...
3,Chainsaw Man Movie: Reze-hen,9.07,4,540,Despite the immediate challenges following bec...,movie,1,"action, fantasy, gore, shounen, urban fantasy",mappa,action fantasy gore shounen urban fantasy ...,action fantasy gore shounen urban fantasy ...,movie movie movie movie movie action fantasy ...,movie movie movie movie movie movie movie movi...
4,Steins;Gate,9.07,5,14,Eccentric scientist Rintarou Okabe has a never...,tv,24,"drama, psychological, sci-fi, suspense, time t...",white fox,drama psychological sci-fi suspense time t...,drama psychological sci-fi suspense time t...,tv tv tv tv tv drama psychological sci-fi s...,tv tv tv tv tv tv tv tv drama psychological ...


## 7. 🏷️ Feature Engineering — Membuat Kolom Tags
Menggabungkan fitur `genres`, `synopsis`, `studios`, dan `media_type` menjadi satu kolom `tags` sebagai representasi setiap anime untuk model TF-IDF.

Strategi pembobotan:
- `genres` diulang 3x → bobot tertinggi
- `synopsis` 200 karakter pertama → menangkap tema cerita
- `studios` dan `media_type` → konteks tambahan

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Buat kolom 'tags' — gabungan semua fitur
def create_tags(row):
    genres = str(row['genres']).replace(',', ' ')
    studios = str(row['studios']).replace(',', ' ')
    media_type = str(row['media_type'])
    synopsis = str(row['synopsis'])[:200]  # ambil 200 karakter pertama
    
    return f"{genres} {genres} {genres} {studios} {media_type} {synopsis}"
    # genres diulang 3x agar bobotnya lebih tinggi

df_combined['tags'] = df_combined.apply(create_tags, axis=1)

# Lowercase semua tags
df_combined['tags'] = df_combined['tags'].str.lower()

print("Sample tags:")
print(df_combined[['title', 'tags']].head(3).to_string())

Sample tags:
                                      title                                                                                                                                                                                                                                                                                                                                                                                                                                                                     tags
0                         Sousou no Frieren                                                                                               adventure  award winning  drama  fantasy  shounen adventure  award winning  drama  fantasy  shounen adventure  award winning  drama  fantasy  shounen madhouse tv during their decade-long quest to defeat the demon king, the members of the hero's party—himmel himself, the priest heiter, the dwarf warrior eisen, and the elven mage frieren—forge bond

## 8. 🤖 TF-IDF Vectorization
Mengubah kolom `tags` menjadi representasi vektor numerik menggunakan TF-IDF.

- `max_features=10.000` — ambil 10.000 kata paling penting
- `ngram_range=(1,2)` — tangkap frasa 2 kata seperti "martial arts", "high school"
- `stop_words='english'` — buang kata umum

In [11]:
# TF-IDF Vectorization
print("Memproses TF-IDF... (mungkin butuh beberapa detik)")

tfidf = TfidfVectorizer(
    max_features=5000,  # ambil 5000 kata paling penting
    stop_words='english'  # buang kata umum seperti "the", "a", "in"
)

tfidf_matrix = tfidf.fit_transform(df_combined['tags'])

print(f"TF-IDF Matrix shape: {tfidf_matrix.shape}")
print("TF-IDF selesai diproses! ✅")

Memproses TF-IDF... (mungkin butuh beberapa detik)
TF-IDF Matrix shape: (25240, 5000)
TF-IDF selesai diproses! ✅


## 9. 🎯 Fungsi Rekomendasi
Membangun fungsi rekomendasi

In [12]:
# Reset index untuk mapping judul
df_combined = df_combined.reset_index(drop=True)
indices = pd.Series(df_combined.index, index=df_combined['title'].str.lower())

def get_recommendations(title, n=10):
    title_lower = title.lower()
    
    # Cek apakah anime ada di dataset
    if title_lower not in indices:
        return None, f"Anime '{title}' tidak ditemukan di dataset."
    
    idx = indices[title_lower]
    
    # Hitung cosine similarity
    sim_scores = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).flatten()
    sim_scores[idx] = 0  # exclude anime itu sendiri
    
    # Ambil top 50 anime paling mirip
    top_indices = sim_scores.argsort()[::-1][:50]
    top_df = df_combined.iloc[top_indices].copy()
    top_df['similarity'] = sim_scores[top_indices]
    
    # TOP PICKS — mirip + rating tinggi + populer
    top_picks = top_df[top_df['mean'] >= 7.0].sort_values(
        by=['mean', 'popularity'], ascending=[False, True]
    ).head(n)
    
    # HIDDEN GEMS — mirip + rating rendah/sedang + kurang populer
    hidden_gems = top_df[top_df['mean'] < 7.0].sort_values(
        by=['similarity'], ascending=False
    ).head(n)
    
    return top_picks, hidden_gems

# Test dengan Naruto
top, hidden = get_recommendations("Naruto")

print("=== TOP PICKS ===")
print(top[['title', 'mean', 'popularity', 'genres']].to_string())
print("\n=== HIDDEN GEMS ===")
print(hidden[['title', 'mean', 'popularity', 'genres']].to_string())

=== TOP PICKS ===
                                     title  mean  popularity                                                  genres
138       Fanren Xiu Xian Zhuan 4th Season  8.54       11168    action, adventure, fantasy, historical, martial arts
187                        Yuu☆Yuu☆Hakusho  8.46         308  action, martial arts, mythology, shounen, supernatural
263             Tunshi Xingkong 3rd Season  8.37        9407        action, adventure, fantasy, martial arts, sci-fi
270             Tunshi Xingkong 4th Season  8.36        8842        action, adventure, fantasy, martial arts, sci-fi
274      Doupo Cangqiong: San Nian Zhi Yue  8.35        6836    action, adventure, fantasy, historical, martial arts
275  Fanren Xiu Xian Zhuan: Xinghai Feichi  8.35       10236    action, adventure, fantasy, historical, martial arts
302              Doupo Cangqiong: Nian Fan  8.32        6666    action, adventure, fantasy, historical, martial arts
337                     Naruto: Shippuuden  8.

In [13]:
def get_recommendations(title, n=10):
    title_lower = title.lower()
    
    if title_lower not in indices:
        return None, f"Anime '{title}' tidak ditemukan di dataset."
    
    idx = indices[title_lower]
    
    # Hitung cosine similarity
    sim_scores = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).flatten()
    sim_scores[idx] = 0
    
    # Ambil top 100 anime paling mirip
    top_indices = sim_scores.argsort()[::-1][:100]
    top_df = df_combined.iloc[top_indices].copy()
    top_df['similarity'] = sim_scores[top_indices]
    
    # Buang anime tanpa rating
    top_df = top_df[top_df['mean'] > 0]
    
    # TOP PICKS — rating >= 7.5, popularity < 5000 (makin kecil makin populer)
    top_picks = top_df[
        (top_df['mean'] >= 7.5) & 
        (top_df['popularity'] > 0)
    ].sort_values(by=['mean', 'similarity'], ascending=[False, False]).head(n)
    
    # HIDDEN GEMS — rating < 7.5, similarity tinggi
    hidden_gems = top_df[
        (top_df['mean'] < 7.5) &
        (top_df['mean'] > 0)
    ].sort_values(by=['similarity', 'mean'], ascending=[False, False]).head(n)
    
    return top_picks, hidden_gems

# Test lagi
top, hidden = get_recommendations("Naruto")

print("=== TOP PICKS ===")
print(top[['title', 'mean', 'popularity', 'genres']].to_string())
print("\n=== HIDDEN GEMS ===")
print(hidden[['title', 'mean', 'popularity', 'genres']].to_string())

=== TOP PICKS ===
                                     title  mean  popularity                                                                genres
138       Fanren Xiu Xian Zhuan 4th Season  8.54       11168                  action, adventure, fantasy, historical, martial arts
173                                Xian Ni  8.49        6881                  action, adventure, fantasy, historical, martial arts
187                        Yuu☆Yuu☆Hakusho  8.46         308                action, martial arts, mythology, shounen, supernatural
263             Tunshi Xingkong 3rd Season  8.37        9407                      action, adventure, fantasy, martial arts, sci-fi
270             Tunshi Xingkong 4th Season  8.36        8842                      action, adventure, fantasy, martial arts, sci-fi
275  Fanren Xiu Xian Zhuan: Xinghai Feichi  8.35       10236                  action, adventure, fantasy, historical, martial arts
274      Doupo Cangqiong: San Nian Zhi Yue  8.35        6836     

In [14]:
# Approach baru — synopsis lebih panjang + bobot lebih seimbang
def create_tags_v2(row):
    genres = str(row['genres']).replace(',', ' ')
    studios = str(row['studios']).replace(',', ' ')
    media_type = str(row['media_type'])
    synopsis = str(row['synopsis'])[:500]  # perbesar jadi 500 karakter
    
    # genres diulang 2x, synopsis lebih panjang
    return f"{genres} {genres} {synopsis} {studios} {media_type}"

df_combined['tags_v2'] = df_combined.apply(create_tags_v2, axis=1)
df_combined['tags_v2'] = df_combined['tags_v2'].str.lower()

# TF-IDF baru dengan lebih banyak fitur
tfidf_v2 = TfidfVectorizer(
    max_features=10000,  # naikkan jadi 10000
    stop_words='english',
    ngram_range=(1, 2)  # tangkap frasa 2 kata seperti "martial arts", "high school"
)

tfidf_matrix_v2 = tfidf_v2.fit_transform(df_combined['tags_v2'])
print(f"TF-IDF v2 Matrix shape: {tfidf_matrix_v2.shape}")

# Test ulang dengan Naruto
indices_v2 = pd.Series(df_combined.index, index=df_combined['title'].str.lower())

def get_recommendations_v2(title, n=10):
    title_lower = title.lower()
    
    if title_lower not in indices_v2:
        return None, f"Anime '{title}' tidak ditemukan."
    
    idx = indices_v2[title_lower]
    sim_scores = cosine_similarity(tfidf_matrix_v2[idx], tfidf_matrix_v2).flatten()
    sim_scores[idx] = 0
    
    top_indices = sim_scores.argsort()[::-1][:100]
    top_df = df_combined.iloc[top_indices].copy()
    top_df['similarity'] = sim_scores[top_indices]
    top_df = top_df[top_df['mean'] > 0]
    
    top_picks = top_df[top_df['mean'] >= 7.5].sort_values(
        by=['mean', 'similarity'], ascending=[False, False]
    ).head(n)
    
    hidden_gems = top_df[top_df['mean'] < 7.5].sort_values(
        by=['similarity', 'mean'], ascending=[False, False]
    ).head(n)
    
    return top_picks, hidden_gems

top, hidden = get_recommendations_v2("Naruto")
print("\n=== TOP PICKS ===")
print(top[['title', 'mean', 'popularity', 'genres']].to_string())
print("\n=== HIDDEN GEMS ===")
print(hidden[['title', 'mean', 'popularity', 'genres']].to_string())

TF-IDF v2 Matrix shape: (25240, 10000)

=== TOP PICKS ===
                                     title  mean  popularity                                                genres
138       Fanren Xiu Xian Zhuan 4th Season  8.54       11168  action, adventure, fantasy, historical, martial arts
173                                Xian Ni  8.49        6881  action, adventure, fantasy, historical, martial arts
263             Tunshi Xingkong 3rd Season  8.37        9407      action, adventure, fantasy, martial arts, sci-fi
270             Tunshi Xingkong 4th Season  8.36        8842      action, adventure, fantasy, martial arts, sci-fi
275  Fanren Xiu Xian Zhuan: Xinghai Feichi  8.35       10236  action, adventure, fantasy, historical, martial arts
274      Doupo Cangqiong: San Nian Zhi Yue  8.35        6836  action, adventure, fantasy, historical, martial arts
302              Doupo Cangqiong: Nian Fan  8.32        6666  action, adventure, fantasy, historical, martial arts
337                   

## 10. 🎯 Revisi Feature Engineering dan Fungsi Rekomendasi

In [133]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import re

def create_tags_v3(row):
    genres = str(row['genres']).replace(',', ' ')
    studios = str(row['studios']).replace(',', ' ')
    media_type = str(row['media_type'])
    synopsis = str(row['synopsis'])[:500]
    
    # media_type diulang 5x — bobot tinggi
    # genres diulang 2x — turunkan sedikit
    # synopsis tetap panjang
    return f"{media_type} {media_type} {media_type} {media_type} {media_type} {genres} {genres} {genres} {genres} {synopsis} {synopsis} {synopsis} {studios}"

df_combined['tags_v3'] = df_combined.apply(create_tags_v3, axis=1)
df_combined['tags_v3'] = df_combined['tags_v3'].str.lower()

tfidf_v3 = TfidfVectorizer(
    max_features=10000,
    stop_words='english',
    ngram_range=(1, 2)
)

tfidf_matrix_v3 = tfidf_v3.fit_transform(df_combined['tags_v3'])
print(f"TF-IDF v3 Matrix shape: {tfidf_matrix_v3.shape}")

def is_sequel(input_title, candidate_title):
    """Cek apakah candidate adalah sequel/prequel/spin-off dari input"""
    input_clean = input_title.lower().strip()
    candidate_clean = candidate_title.lower().strip()
    
    # Ambil kata-kata utama dari judul input (minimal 3 karakter)
    input_words = set(w for w in input_clean.split() if len(w) >= 3)
    candidate_words = set(w for w in candidate_clean.split() if len(w) >= 3)
    
    # Kalau lebih dari 50% kata input ada di candidate, anggap sequel
    if len(input_words) == 0:
        return False
    overlap = len(input_words & candidate_words) / len(input_words)
    return overlap >= 0.5

def get_recommendations_v3(title, n=10):
    title_lower = title.lower()
    
    if title_lower not in indices_v2:
        return None, f"Anime '{title}' tidak ditemukan."
    
    idx = indices_v2[title_lower]
    sim_scores = cosine_similarity(tfidf_matrix_v3[idx], tfidf_matrix_v3).flatten()
    sim_scores[idx] = 0
    
    top_indices = sim_scores.argsort()[::-1][:100]
    top_df = df_combined.iloc[top_indices].copy()
    top_df['similarity'] = sim_scores[top_indices]
    top_df = top_df[top_df['mean'] > 0]

    # Filter sequel/prequel
    top_df = top_df[~top_df['title'].apply(lambda x: is_sequel(title, x))]
    
    # Scoring gabungan mean + rank
    # rank: makin kecil makin bagus, kita balik jadi score
    max_rank = df_combined['rank'].max()
    top_df['rank_score'] = 1 - (top_df['rank'] / max_rank)
    top_df['combined_score'] = (top_df['mean'] / 10 * 0.6) + (top_df['rank_score'] * 0.4)
    
    # TOP PICKS
    top_picks = top_df[top_df['mean'] >= 7.5].sort_values(
        by=['combined_score', 'similarity'], ascending=[False, False]
    ).head(n)
    
    # HIDDEN GEMS — exclude anime yang sudah ada di top picks
    top_picks_indices = set(top_picks.index)
    hidden_gems = top_df[
        (top_df['mean'] >= 7.0) &
        (top_df['popularity'] > 2000) &
        (~top_df.index.isin(top_picks_indices))  # exclude top picks!
    ].sort_values(
        by=['similarity', 'mean'], ascending=[False, False]
    ).head(n * 2)  # ambil 2x lipat untuk "more" button
    
    return top_picks, hidden_gems

top, hidden = get_recommendations_v3("death note")
print("\n=== TOP PICKS ===")
print(top[['title', 'mean', 'rank', 'popularity', 'genres', 'media_type']].to_string())
print("\n=== HIDDEN GEMS ===")
print(hidden[['title', 'mean', 'rank', 'popularity', 'genres', 'media_type']].to_string())

TF-IDF v3 Matrix shape: (25240, 10000)

=== TOP PICKS ===
                                        title  mean  rank  popularity                                                                                     genres media_type
94   Jujutsu Kaisen: Shimetsu Kaiyuu - Zenpen  8.62    95         518                                                              action, shounen, supernatural         tv
181                         Summertime Render  8.47   182         358                                      mystery, shounen, supernatural, suspense, time travel         tv
182                     Yakusoku no Neverland  8.47   183          37                                        mystery, psychological, shounen, survival, suspense         tv
187                           Yuu☆Yuu☆Hakusho  8.46   188         308                                     action, martial arts, mythology, shounen, supernatural         tv
222                                  Mononoke  8.41   223         750  adult cast,

## HIDE

In [72]:
import re

def is_sequel(input_title, candidate_title):
    """Cek apakah candidate adalah sequel/prequel/spin-off dari input"""
    input_clean = input_title.lower().strip()
    candidate_clean = candidate_title.lower().strip()
    
    # Ambil kata-kata utama dari judul input (minimal 3 karakter)
    input_words = set(w for w in input_clean.split() if len(w) >= 3)
    candidate_words = set(w for w in candidate_clean.split() if len(w) >= 3)
    
    # Kalau lebih dari 50% kata input ada di candidate, anggap sequel
    if len(input_words) == 0:
        return False
    overlap = len(input_words & candidate_words) / len(input_words)
    return overlap >= 0.5

def get_recommendations_v5(title, n=10):
    title_lower = title.lower()
    
    if title_lower not in indices_v2:
        return None, f"Anime '{title}' tidak ditemukan."
    
    idx = indices_v2[title_lower]
    sim_scores = cosine_similarity(tfidf_matrix_v3[idx], tfidf_matrix_v3).flatten()
    sim_scores[idx] = 0
    
    # Naikkan pool ke 300
    top_indices = sim_scores.argsort()[::-1][:300]
    top_df = df_combined.iloc[top_indices].copy()
    top_df['similarity'] = sim_scores[top_indices]
    
    # Buang anime tanpa rating
    top_df = top_df[top_df['mean'] > 0]
    
    # Filter sequel/prequel
    top_df = top_df[~top_df['title'].apply(lambda x: is_sequel(title, x))]
    
    # Combined score
    max_rank = df_combined[df_combined['rank'] > 0]['rank'].max()
    top_df['rank_score'] = top_df['rank'].apply(lambda x: 1 - (x / max_rank) if x > 0 else 0)
    top_df['combined_score'] = (top_df['mean'] / 10 * 0.6) + (top_df['rank_score'] * 0.4)
    
    # TOP PICKS — turunkan threshold ke 7.0
    top_picks = top_df[top_df['mean'] >= 7.0].sort_values(
        by=['combined_score', 'similarity'], ascending=[False, False]
    ).head(n)
    
    # HIDDEN GEMS
    hidden_gems = top_df[
        (top_df['mean'] < 7.0) &
        (top_df['mean'] > 0)
    ].sort_values(
        by=['similarity', 'mean'], ascending=[False, False]
    ).head(n)
    
    return top_picks, hidden_gems

# Test beberapa anime
for test_title in ["death note"]:
    top, hidden = get_recommendations_v5(test_title)
    print(f"\n{'='*60}")
    print(f"INPUT: {test_title}")
    print(f"{'='*60}")
    print("TOP PICKS:")
    print(top[['title', 'mean', 'genres', 'media_type']].to_string())
    print("\nHIDDEN GEMS:")
    print(hidden[['title', 'mean', 'genres', 'media_type']].to_string())


INPUT: death note
TOP PICKS:
                                                     title  mean                                                                                     genres media_type
2                         Fullmetal Alchemist: Brotherhood  9.11                                       action, adventure, drama, fantasy, military, shounen         tv
46                                             Ikoku Nikki  8.76                                                                               drama, josei         tv
66              Bleach: Sennen Kessen-hen - Ketsubetsu-tan  8.70                                                   action, adventure, shounen, supernatural         tv
67                               Jujutsu Kaisen 2nd Season  8.70                                                action, gore, school, shounen, supernatural         tv
70                           Kimetsu no Yaiba: Yuukaku-hen  8.69                                                  action, historical, s

## 11. 🎯 Final Fungsi Rekomendasi

In [73]:
def get_recommendations_final(title, n=10):
    title_lower = title.lower()
    
    if title_lower not in indices_v2:
        return None, f"Anime '{title}' tidak ditemukan."
    
    idx = indices_v2[title_lower]
    sim_scores = cosine_similarity(tfidf_matrix_v3[idx], tfidf_matrix_v3).flatten()
    sim_scores[idx] = 0
    
    top_indices = sim_scores.argsort()[::-1][:500]
    top_df = df_combined.iloc[top_indices].copy()
    top_df['similarity'] = sim_scores[top_indices]
    
    # Buang anime tanpa rating
    top_df = top_df[top_df['mean'] > 0]
    
    # Filter sequel
    top_df = top_df[~top_df['title'].apply(lambda x: is_sequel(title, x))]
    
    # Combined score
    max_rank = df_combined[df_combined['rank'] > 0]['rank'].max()
    top_df['rank_score'] = top_df['rank'].apply(lambda x: 1 - (x / max_rank) if x > 0 else 0)
    top_df['combined_score'] = (top_df['mean'] / 10 * 0.6) + (top_df['rank_score'] * 0.4)
    
    # TOP PICKS — populer (popularity <= 2000) + rating >= 7.0
    top_picks = top_df[
        (top_df['mean'] >= 7.0) &
        (top_df['popularity'] <= 2000)
    ].sort_values(
        by=['combined_score', 'similarity'], ascending=[False, False]
    ).head(n * 2)  # ambil 2x lipat untuk "more" button
    
    # Kalau kurang dari n, relax filter popularity
    if len(top_picks) < n:
        extra = top_df[
            (top_df['mean'] >= 7.0) &
            (~top_df.index.isin(top_picks.index))
        ].sort_values(by=['combined_score'], ascending=False).head(n - len(top_picks))
        top_picks = pd.concat([top_picks, extra])
    
    # HIDDEN GEMS — exclude anime yang sudah ada di top picks
    top_picks_indices = set(top_picks.index)
    hidden_gems = top_df[
        (top_df['mean'] >= 7.0) &
        (top_df['popularity'] > 2000) &
        (~top_df.index.isin(top_picks_indices))  # exclude top picks!
    ].sort_values(
        by=['similarity', 'mean'], ascending=[False, False]
    ).head(n * 2)  # ambil 2x lipat untuk "more" button
    
    return top_picks, hidden_gems

# Test
for test_title in ["death note"]:
    top, hidden = get_recommendations_final(test_title)
    print(f"\n{'='*60}")
    print(f"INPUT: {test_title}")
    print(f"{'='*60}")
    print("TOP PICKS (first 10):")
    if top is None or len(top) == 0:
        print("Tidak ada hasil")
    else:
        print(top.head(10)[['title', 'mean', 'popularity', 'media_type']].to_string())
    print("\nHIDDEN GEMS (first 10):")
    if hidden is None or len(hidden) == 0:
        print("Tidak ada hasil")
    else:
        print(hidden.head(10)[['title', 'mean', 'popularity', 'media_type']].to_string())


INPUT: death note
TOP PICKS (first 10):
                                                     title  mean  popularity media_type
2                         Fullmetal Alchemist: Brotherhood  9.11           3         tv
4                                              Steins;Gate  9.07          14         tv
66              Bleach: Sennen Kessen-hen - Ketsubetsu-tan  8.70         647         tv
67                               Jujutsu Kaisen 2nd Season  8.70         107         tv
70                           Kimetsu no Yaiba: Yuukaku-hen  8.69          72         tv
74                 Bleach: Sennen Kessen-hen - Soukoku-tan  8.68         934         tv
80   Kimetsu no Yaiba Movie 1: Mugenjou-hen - Akaza Sairai  8.67         725      movie
94                Jujutsu Kaisen: Shimetsu Kaiyuu - Zenpen  8.62         518         tv
137                                 Yojouhan Shinwa Taikei  8.55         528         tv
165                                         Jujutsu Kaisen  8.50          11   

## 12. 📈 Evaluasi Model

In [107]:
def evaluate_genre_overlap(test_titles, n=10):
    results = []
    
    for title in test_titles:
        top, hidden = get_recommendations_final(title, n=n)
        
        # Ambil genre input anime
        input_anime = df_combined[df_combined['title'].str.lower() == title.lower()]
        if len(input_anime) == 0:
            continue
            
        input_genres = set(input_anime.iloc[0]['genres'].split(', '))
        
        # Hitung genre overlap untuk top picks
        if top is not None and len(top) > 0:
            overlaps = []
            for _, row in top.head(n).iterrows():
                rec_genres = set(str(row['genres']).split(', '))
                overlap = len(input_genres & rec_genres) / len(input_genres) if len(input_genres) > 0 else 0
                overlaps.append(overlap)
            avg_overlap = sum(overlaps) / len(overlaps)
        else:
            avg_overlap = 0
            
        results.append({
            'anime': title,
            'input_genres': ', '.join(input_genres),
            'top_picks_count': len(top) if top is not None else 0,
            'hidden_gems_count': len(hidden) if hidden is not None else 0,
            'genre_overlap_score': round(avg_overlap * 100, 2)
        })
    
    return pd.DataFrame(results)

# Test dengan 10 anime beragam genre
test_titles = [
    "Naruto", "Death Note", "Shingeki no Kyojin",
    "Fullmetal Alchemist: Brotherhood", "Steins;Gate",
    "Violet Evergarden", "Bungou Stray Dogs",
    "One Punch Man", "Hunter x Hunter", "Cowboy Bebop"
]

eval_df = evaluate_genre_overlap(test_titles)
print("=== EVALUASI MODEL ===\n")
print(eval_df.to_string(index=False))
print(f"\nRata-rata Genre Overlap Score: {eval_df['genre_overlap_score'].mean():.2f}%")
print(f"Min: {eval_df['genre_overlap_score'].min():.2f}%")
print(f"Max: {eval_df['genre_overlap_score'].max():.2f}%")

=== EVALUASI MODEL ===

                           anime                                                                 input_genres  top_picks_count  hidden_gems_count  genre_overlap_score
                          Naruto                            adventure, fantasy, shounen, action, martial arts               20                 20                62.00
                      Death Note                               suspense, psychological, supernatural, shounen               20                 20                47.50
              Shingeki no Kyojin    suspense, survival, drama, military, action, shounen, gore, award winning               20                 20                33.75
Fullmetal Alchemist: Brotherhood                         adventure, fantasy, drama, military, action, shounen               20                 20                58.33
                     Steins;Gate                          suspense, psychological, drama, sci-fi, time travel               20               

In [108]:
def evaluate_full(test_titles, n=10):
    results = []
    
    for title in test_titles:
        top, hidden = get_recommendations_final(title, n=n)
        
        input_anime = df_combined[df_combined['title'].str.lower() == title.lower()]
        if len(input_anime) == 0:
            continue
            
        input_genres = set(input_anime.iloc[0]['genres'].split(', '))
        input_media_type = input_anime.iloc[0]['media_type']
        input_idx = indices_v2[title.lower()]
        
        top_picks_count = len(top.head(n)) if top is not None and len(top) > 0 else 0
        hidden_gems_count = len(hidden.head(n)) if hidden is not None and len(hidden) > 0 else 0
        
        # 1. Genre Overlap Score
        genre_overlaps = []
        media_type_match = []
        sim_scores_list = []
        
        if top is not None and len(top) > 0:
            for _, row in top.head(n).iterrows():
                # Genre overlap
                rec_genres = set(str(row['genres']).split(', '))
                overlap = len(input_genres & rec_genres) / len(input_genres) if len(input_genres) > 0 else 0
                genre_overlaps.append(overlap)
                
                # Media type consistency
                media_type_match.append(1 if row['media_type'] == input_media_type else 0)
                
                # Cosine similarity
                sim_scores_list.append(row['similarity'])
        
        results.append({
            'anime': title,
            'top_picks': top_picks_count,
            'hidden_gems': hidden_gems_count,
            'genre_overlap_%': round(sum(genre_overlaps)/len(genre_overlaps)*100, 2) if genre_overlaps else 0,
            'media_type_consistency_%': round(sum(media_type_match)/len(media_type_match)*100, 2) if media_type_match else 0,
            'avg_cosine_similarity': round(sum(sim_scores_list)/len(sim_scores_list), 4) if sim_scores_list else 0
        })
    
    df_eval = pd.DataFrame(results)
    
    print("=== EVALUASI MODEL LENGKAP ===\n")
    print(df_eval.to_string(index=False))
    print(f"\n{'='*60}")
    print(f"SUMMARY:")
    print(f"Rata-rata Genre Overlap        : {df_eval['genre_overlap_%'].mean():.2f}%")
    print(f"Rata-rata Media Type Consistency: {df_eval['media_type_consistency_%'].mean():.2f}%")
    print(f"Rata-rata Cosine Similarity    : {df_eval['avg_cosine_similarity'].mean():.4f}")
    
    return df_eval

eval_df = evaluate_full(test_titles)

=== EVALUASI MODEL LENGKAP ===

                           anime  top_picks  hidden_gems  genre_overlap_%  media_type_consistency_%  avg_cosine_similarity
                          Naruto         10           10            62.00                      90.0                 0.1412
                      Death Note         10           10            47.50                      70.0                 0.0802
              Shingeki no Kyojin         10           10            33.75                      70.0                 0.0967
Fullmetal Alchemist: Brotherhood         10           10            58.33                      90.0                 0.0965
                     Steins;Gate         10           10            64.00                      30.0                 0.1307
               Violet Evergarden         10           10            40.00                      80.0                 0.0728
               Bungou Stray Dogs         10           10            37.14                      70.0        

In [130]:
# Ambil 50 anime populer secara acak dari berbagai genre
sample_titles = df_combined[
    (df_combined['mean'] > 7.0) & 
    (df_combined['popularity'] <= 3000)
].sample(50, random_state=42)['title'].tolist()

print("50 anime yang akan dievaluasi:")
for i, t in enumerate(sample_titles, 1):
    print(f"{i}. {t}")

50 anime yang akan dievaluasi:
1. Yamada-kun to Lv999 no Koi wo Suru
2. Heppoko Jikken Animation Excel♥Saga
3. One Piece Movie 01
4. Love Hina
5. Diamond no Ace
6. Boku no Hero Academia
7. Fullmetal Alchemist: Brotherhood - 4-Koma Theater
8. Dragon Ball Z Movie 09: Ginga Girigiri!! Bucchigiri no Sugoi Yatsu
9. JoJo no Kimyou na Bouken Part 5: Ougon no Kaze
10. Josee to Tora to Sakana-tachi
11. Zoku Owarimonogatari
12. Dragon Ball
13. Neon Genesis Evangelion: Death & Rebirth
14. Kore wa Zombie desu ka? of the Dead: Hai, Minotake ni Attemasu
15. Kimi to, Nami ni Noretara
16. Bungou Stray Dogs 2nd Season
17. xxxHOLiC Movie: Manatsu no Yoru no Yume
18. Tokyo Ghoul: "Pinto"
19. Utawarerumono
20. Rakudai Kishi no Cavalry
21. Xian Wang de Richang Shenghuo 3
22. Digimon Adventure tri. 1: Saikai
23. Amagami SS+ Plus
24. Love Stage!!
25. Lord El-Melloi II Sei no Jikenbo: Rail Zeppelin Grace Note
26. Haikyuu!! To the Top
27. Keroro Gunsou
28. Overlord IV
29. New Panty & Stocking with Garterbelt
3

In [134]:
eval_df_50 = evaluate_full(sample_titles, n=10)

=== EVALUASI MODEL LENGKAP ===

                                                             anime  top_picks  hidden_gems  genre_overlap_%  media_type_consistency_%  avg_cosine_similarity
                                Yamada-kun to Lv999 no Koi wo Suru         10           10            25.00                      80.0                 0.1032
                               Heppoko Jikken Animation Excel♥Saga         10           10            60.00                      70.0                 0.1032
                                                One Piece Movie 01         10           10            86.67                       0.0                 0.1115
                                                         Love Hina         10           10            56.67                      80.0                 0.0687
                                                    Diamond no Ace         10           10            60.00                      80.0                 0.1882
                          

## 13. 💾 Simpan Model

In [147]:
import pickle

with open('../data/tfidf_model.pkl', 'wb') as f:
    pickle.dump(tfidf_v3, f)

with open('../data/tfidf_matrix.pkl', 'wb') as f:
    pickle.dump(tfidf_matrix_v3, f)

with open('../data/indices.pkl', 'wb') as f:
    pickle.dump(indices_v2, f)

df_combined.to_csv('../data/anime_cleaned.csv', index=False)

print("✅ Semua file berhasil disimpan!")
print("- data/tfidf_model.pkl")
print("- data/tfidf_matrix.pkl")
print("- data/indices.pkl")
print("- data/anime_cleaned.csv")

✅ Semua file berhasil disimpan!
- data/tfidf_model.pkl
- data/tfidf_matrix.pkl
- data/indices.pkl
- data/anime_cleaned.csv


In [135]:
import pickle

with open('../data/tfidf_model.pkl', 'wb') as f:
    pickle.dump(tfidf_v3, f)

with open('../data/tfidf_matrix.pkl', 'wb') as f:
    pickle.dump(tfidf_matrix_v3, f)

with open('../data/indices.pkl', 'wb') as f:
    pickle.dump(indices_v2, f)

print("✅ Model baru berhasil disimpan!")

✅ Model baru berhasil disimpan!
